# 生成式AI應用開發 第 12 週｜多模態應用：Vision API 與圖片理解

<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>🎯 本週目標</b>：把輸入從文字／文件延伸到<b>圖片</b>。用 OpenAI Responses API 的 <code>input_image</code> 做圖片描述、圖片問答、截圖檢查，以及收據／表單的<b>結構化抽取</b>，並做成 Streamlit 圖片理解工具。<br>使用工具：<b>OpenAI Responses API（input_text + input_image）+ JSON Schema + Pillow</b>。
</div>

> 🧩 <b>本檔為 Claude Code 產出的學生版</b>。helper 命名（<code>detect_image_mime</code>／<code>validate_image</code>／<code>image_to_data_url</code>／<code>build_task_prompt</code>／<code>require_completed_response</code>／<code>analyze_image</code>／<code>RECEIPT_SCHEMA</code>）與 Codex 版 <code>week12_vision_app/</code> 一致，兩版可互換對照。

<div style="background-color:#fdeeee; border-left:6px solid #d9534f; padding:12px 16px; border-radius:6px;">
<b>⚠️ 安全紅線</b>：圖片會傳送到外部 API。課堂只用<b>自己製作或明確授權、且不含敏感資料</b>的圖片；<b>不上傳</b>身分證、成績、病歷、醫療影像、金融資料、公司機密、未授權人像或 CAPTCHA。API key 只能放在 Colab Secrets、<code>.env</code> 或 Streamlit Secrets。
</div>

## 0️⃣ 本週學習目標與三小時流程

完成本週後，你應能：

1. 說明圖片如何以 URL、Base64 data URL 或 file ID 進入模型，以及本週為何選 data URL。
2. 依**檔案內容**驗證圖片格式、大小、尺寸與動態 GIF，不只相信副檔名（`validate_image`）。
3. 在同一個 user message 裡組合 `input_text` 與 `input_image`（`build_request`）。
4. 為描述、問答、截圖、抽取四種任務設計「證據」與「推論」分開的 prompt（`build_task_prompt`）。
5. 用 JSON Schema（strict）取得程式可讀的收據／表單欄位（`RECEIPT_SCHEMA`）。
6. 分流拒答、未完成、空輸出與解析失敗（`require_completed_response`），並理解成本與隱私風險。

| 時間 | 內容 | 產出 |
|---|---|---|
| 35 分 | 多模態心智模型、格式、detail、成本與限制 | 看懂圖片輸入資料流 |
| 50 分 | 驗證、Base64、Responses API、錯誤分流 | 完成最小圖片問答 |
| 45 分 | Prompt 設計與 Structured Outputs | 完成收據／表單抽取 |
| 40 分 | 練習 A～D 與失敗案例 | 完成 helper 與基本評估 |
| 10 分 | Streamlit 專案說明與第 13 週銜接 | 能啟動圖片理解 App |

## 📌 這週在整個課程的位置

| 週次 | 產出 | 本週如何銜接 |
|---|---|---|
| 第 6 週 | Structured Outputs：JSON Schema 抽取 | ← 同一套 schema 技巧，輸入換成圖片 |
| 第 9～11 週 | 文件處理、檢索、RAG：輸入驗證與「只依證據回答」 | ← 驗證 bytes、abstain 的觀念延續 |
| **第 12 週** | **圖片輸入 → 描述／問答／截圖／抽取** | 本週 |
| 第 13 週 | Function Calling：把函式交給模型選用 | → 本週的 `validate_image` / `analyze_image` 就是候選工具 |

<div style="background-color:#f0fff4; border-left:6px solid #5cb85c; padding:12px 16px; border-radius:6px;">
<b>一句話</b>：第 11 週讓模型「只依文件回答」；第 12 週讓模型「只依<b>圖片可見內容</b>回答」，看不清楚就說看不清楚。
</div>

## 1️⃣ 多模態心智模型

一個請求可以同時包含多種 content：文字用 `input_text`，圖片用 `input_image`。圖片可以用三種方式進入模型：

| 方式 | 適合情境 | 本週是否使用 |
|---|---|---|
| 公開 URL | 圖片已在網路上 | 否（課堂圖片不該公開） |
| **Base64 data URL** | 圖片 bytes 在手上（上傳器給的就是 bytes） | **是** |
| file ID（先上傳到 Files API） | 同一張圖要重複用很多次 | 否（選讀） |

資料流：

```
圖片 bytes ─▶ validate_image（本機驗證，不花錢）
           ─▶ image_to_data_url（Base64）
           ─▶ build_request（input_text + input_image + 可選 JSON Schema）
           ─▶ Responses API ─▶ require_completed_response（分流拒答／未完成／空輸出）
           ─▶ 文字或 JSON ─▶ 人工核對
```

<div style="background-color:#fff8e6; border-left:6px solid #f0ad4e; padding:12px 16px; border-radius:6px;">
<b>成本與限制</b>：圖片會換算成輸入 token；張數、尺寸、<code>detail</code> 都可能提高成本。Vision 可能看錯小字、數字、物件數量、圖表與空間關係；<b>重要結果一定回看原圖</b>。
</div>

## 2️⃣ 環境與課堂測試圖片

本 Notebook 可**離線執行**（不需 API key）：驗證、Base64、prompt、request 預覽與離線示範都不花錢；
只有 `RUN_PAID_API = True` 的 cell 才會真的呼叫 Vision API。

下一格會用 Pillow 畫一張**完全虛構**的英文收據，以及同一張的模糊旋轉版；課堂不使用真實收據。

> 在 Colab 或全新環境第一次執行時，取消 `%pip install` 的註解。

In [ ]:
# Colab / 新環境第一次執行時，取消下一行註解安裝套件。
# %pip install -q openai python-dotenv pillow

from __future__ import annotations

import base64
from io import BytesIO
import json
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from IPython.display import display
from PIL import Image, ImageDraw, ImageFilter, UnidentifiedImageError


DEFAULT_MODEL = 'gpt-5.4-mini'
MAX_FILE_BYTES = 8 * 1024 * 1024
MAX_IMAGE_PIXELS = 20_000_000
SUPPORTED_MIME_TYPES = {"image/png", "image/jpeg", "image/webp", "image/gif"}
TASK_MODES = ('圖片描述', '圖片問答', '截圖檢查', '收據／表單抽取')
DETAIL_LEVELS = ('auto', 'low', 'high')
OFFLINE_DEMO_NOTICE = '（離線示範：這不是模型分析結果，只用來驗證流程與畫面）'
SYSTEM_INSTRUCTIONS = (
    "你是謹慎的圖片理解助理。回答使用繁體中文；區分圖片直接證據與推論；"
    "無法確認的文字、數字、人物身分或情境不可猜測。"
)

# 付費 API 預設關閉。確認已設定測試用 API key 後，再改成 True。
RUN_PAID_API = False


def get_secret(name: str, default: str | None = None) -> str | None:
    """優先讀 Colab Secrets，其次 .env / 環境變數；不把秘密印出。"""
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    load_dotenv()
    return os.getenv(name, default)


def create_client():
    """建立 OpenAI client；缺少金鑰時先轉成可操作的課堂錯誤。"""
    api_key = get_secret("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("找不到 OPENAI_API_KEY，請先設定 `.env` 或 Streamlit Secrets。")
    from openai import OpenAI

    return OpenAI(api_key=api_key)

MODEL = get_secret("OPENAI_MODEL", DEFAULT_MODEL) or DEFAULT_MODEL
print(f"模型：{MODEL}｜付費 API：{RUN_PAID_API}｜API key 已設定：{bool(get_secret('OPENAI_API_KEY'))}")

In [ ]:
# 建立完全虛構、無個資的課堂收據（清楚版 + 模糊旋轉版），避免課堂範例含真實資料。
CLEAR_PATH = Path("demo_receipt.png")
BLURRY_PATH = Path("demo_receipt_blurry.png")

demo_lines = [
    "DEMO CAFE - CLASSROOM FIXTURE",
    "Date: 2026-09-16",
    "--------------------------------",
    "Coffee          2 x 80      160",
    "Sandwich        1 x 95       95",
    "Notebook        1 x 45       45",
    "--------------------------------",
    "TOTAL TWD                    300",
    "This is fictional test data.",
]

receipt = Image.new("RGB", (900, 1100), "white")
draw = ImageDraw.Draw(receipt)
y = 100
for line in demo_lines:
    draw.text((90, y), line, fill="black", font_size=32)
    y += 90
receipt.save(CLEAR_PATH, format="PNG")

# 模糊 + 旋轉：模擬手機拍歪、失焦的收據，用來觀察模型「看不清楚時」的行為。
receipt.rotate(7, expand=True, fillcolor="white").filter(ImageFilter.GaussianBlur(radius=3)).save(BLURRY_PATH, format="PNG")

clear_bytes = CLEAR_PATH.read_bytes()
blurry_bytes = BLURRY_PATH.read_bytes()
print(f"已建立：{CLEAR_PATH.resolve()}（{len(clear_bytes)} bytes）")
print(f"已建立：{BLURRY_PATH.resolve()}（{len(blurry_bytes)} bytes）")

In [ ]:
demo_image = Image.open(CLEAR_PATH)
display(demo_image.resize((450, 550)))
print("尺寸：", demo_image.size, "格式：", demo_image.format)

## 3️⃣ 格式驗證：不要只相信副檔名

OpenAI 官方支援 PNG、JPEG、WEBP 與非動態 GIF。`st.file_uploader(type=[...])` 只看副檔名，
把文字檔改名成 `.png` 一樣能通過；所以要看**檔案開頭的 signature（magic bytes）**。

<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>detect_image_mime</b> 是 given：PNG 以 <code>89 50 4E 47</code> 開頭、JPEG 以 <code>FF D8 FF</code>、GIF 以 <code>GIF87a/GIF89a</code>、WEBP 是 <code>RIFF....WEBP</code>。
</div>

In [ ]:
def detect_image_mime(file_bytes: bytes) -> str:
    """依檔案 signature（magic bytes）判斷圖片格式，不只相信副檔名或瀏覽器 MIME。

    參數：
        file_bytes: 圖片原始 bytes。

    回傳：
        "image/png"、"image/jpeg"、"image/gif" 或 "image/webp"。

    可能錯誤：
        ValueError: 不是支援的四種格式。

    教學重點：
        上傳器的 type= 篩選只看副檔名；把 .txt 改名成 .png 一樣能通過。
    """
    if file_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        return "image/png"
    if file_bytes.startswith(b"\xff\xd8\xff"):
        return "image/jpeg"
    if file_bytes.startswith((b"GIF87a", b"GIF89a")):
        return "image/gif"
    if len(file_bytes) >= 12 and file_bytes.startswith(b"RIFF") and file_bytes[8:12] == b"WEBP":
        return "image/webp"
    raise ValueError("檔案內容不是支援的 PNG、JPEG、WEBP 或非動態 GIF 圖片。")


print("清楚版：", detect_image_mime(clear_bytes))
# 把文字檔改副檔名成 .png 一樣騙得過 type= 篩選，但騙不過 signature 檢查
try:
    detect_image_mime(b"this is not an image, just renamed to .png")
except ValueError as exc:
    print("假圖片被擋下：", exc)

## 🔧 核心技能 1：`validate_image` — 呼叫付費 API 前的本機把關

除了格式，還要檢查：空檔、8 MB 上限、Pillow 能否解碼、像素量（避免超大圖）、動態 GIF。
驗證通過才回傳 metadata（UI 會拿來顯示尺寸與大小）。

<div style="background-color:#fff8e6; border-left:6px solid #f0ad4e; padding:12px 16px; border-radius:6px;">
<b>為什麼要在本機先驗證？</b>①壞檔送出去一樣計費；②錯誤訊息在本機更清楚；③避免把超大圖丟進 API 造成非預期成本。
</div>

In [ ]:
def validate_image(file_bytes: bytes) -> dict[str, Any]:
    """在呼叫付費 API 前檢查大小、格式、影格與像素量，回傳可供 UI 顯示的 metadata。

    參數／回傳同教師版說明：dict 含 mime_type、format、width、height、megabytes。

    TODO（核心技能 1）：
      1. 空 bytes → raise ValueError；超過 MAX_FILE_BYTES → raise ValueError。
      2. 用 detect_image_mime(file_bytes) 判斷格式（不要看副檔名）。
      3. 用 Pillow 讀取：with Image.open(BytesIO(file_bytes)) as image 取得
         image.size、image.format、getattr(image, "is_animated", False)，並呼叫 image.verify()；
         解碼失敗（UnidentifiedImageError / OSError / SyntaxError）→ raise ValueError。
      4. 寬 × 高 > MAX_IMAGE_PIXELS → raise ValueError；GIF 且 is_animated → raise ValueError。
      5. 回傳 metadata dict（megabytes 取到小數 3 位）。
    提示：目前只做最少檢查並回傳骨架，讓下游 demo 不會壞；完成後跑 run_local_checks() 驗收。
    """
    if not file_bytes:
        raise ValueError("圖片內容是空的，請重新選擇檔案。")
    # TODO: 補上大小、格式、Pillow 解碼、像素量與動態 GIF 檢查
    return {
        "mime_type": detect_image_mime(file_bytes),
        "format": None,
        "width": 0,
        "height": 0,
        "megabytes": round(len(file_bytes) / (1024 * 1024), 3),
    }


print("validate_image 已定義（骨架）")

## 4️⃣ Base64 data URL

Base64 把 bytes 轉成可放進 JSON 的文字，長度約為原始 bytes 的 4/3。
格式：`data:<mime>;base64,<編碼內容>`。**不要把整串印出來或寫進 log**，會洗版也可能外洩圖片內容。

In [ ]:
def image_to_data_url(file_bytes: bytes, mime_type: str) -> str:
    """把圖片 bytes 轉成 Responses API `input_image` 可用的 Base64 data URL。"""
    if mime_type not in SUPPORTED_MIME_TYPES:
        raise ValueError("不支援的圖片 MIME type。")
    encoded = base64.b64encode(file_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


demo_data_url = image_to_data_url(clear_bytes, detect_image_mime(clear_bytes))
print(demo_data_url[:50] + "...")
print("data URL 字元數：", len(demo_data_url), "（約為原始 bytes 的 4/3 倍；不要把整串印出來或存進 log）")

## 5️⃣ Responses API 的圖片輸入與回應分流

`input` 是訊息陣列；同一個 user message 的 `content` 同時放 `input_text` 與 `input_image`。
回傳可先讀 SDK 的 `output_text`，但正式 App 還要檢查 **refusal** 與 **status**，不能假設每次都有完整文字。

```python
client.responses.create(
    model=MODEL,
    instructions="...",
    input=[{"role": "user", "content": [
        {"type": "input_text", "text": prompt},
        {"type": "input_image", "image_url": data_url, "detail": "auto"},
    ]}],
    store=False,
)
```

<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>require_completed_response</b>（given）依序處理：拒答（refusal）→ 未完成（長度限制／內容過濾）→ 狀態異常 → 空輸出，最後才回傳文字。<code>store=False</code> 表示不保存回應供後續取回，<b>不代表圖片沒有傳送</b>。
</div>

In [ ]:
def get_refusal_reason(response: Any) -> str | None:
    """從 Responses API 的 output/content 找出拒答原因；拒答不一定出現在 output_text。"""
    for output_item in getattr(response, "output", []) or []:
        content = (
            output_item.get("content", [])
            if isinstance(output_item, dict)
            else getattr(output_item, "content", [])
        )
        for content_item in content or []:
            refusal = (
                content_item.get("refusal")
                if isinstance(content_item, dict)
                else getattr(content_item, "refusal", None)
            )
            if refusal:
                return str(refusal)
    return None


def require_completed_response(response: Any) -> str:
    """只接受已完成且有文字的回應，避免把拒答、半成品或空輸出當成正式結果。

    可能錯誤：
        RuntimeError: 拒答、未完成（長度限制／內容過濾）、狀態異常、沒有文字。
    """
    refusal = get_refusal_reason(response)
    if refusal:
        raise RuntimeError(f"AI 拒絕處理這張圖片：{refusal}")

    status = getattr(response, "status", None)
    if status == "incomplete":
        details = getattr(response, "incomplete_details", None)
        reason = details.get("reason") if isinstance(details, dict) else getattr(details, "reason", None)
        if reason == "max_output_tokens":
            raise RuntimeError("AI 回應因輸出長度限制而未完成，請縮小任務範圍。")
        if reason == "content_filter":
            raise RuntimeError("圖片或問題觸發內容過濾，請改用合適的課堂測試資料。")
        raise RuntimeError("AI 回應未完成，請稍後再試。")
    if status not in (None, "completed"):
        raise RuntimeError(f"AI 回應未成功（狀態：{status}）。")

    output_text = (getattr(response, "output_text", None) or "").strip()
    if not output_text:
        raise RuntimeError("AI 沒有回傳可顯示的文字結果。")
    return output_text


print("get_refusal_reason / require_completed_response 已定義")

## 🔧 核心技能 2：`build_task_prompt` — 把「證據」與「推論」分開

只問「這張圖是什麼？」容易得到過度概括、甚至編造的答案。可靠的圖片 prompt 會指定：

1. **任務**是描述、問答、截圖檢查還是抽取；
2. **輸出順序**（先可見文字，再狀態，再推論）；
3. **不可猜測的項目**（日期、數字、身分）；
4. **看不清楚時怎麼回**（說明不足、用 null）。

<div style="background-color:#fdeeee; border-left:6px solid #d9534f; padding:12px 16px; border-radius:6px;">
<b>降低幻覺</b>：這是第 11 週「只依文件回答、找不到就說找不到」的圖片版。四種模式的 prompt 都要求「無法從圖片確認就直接說不足」。
</div>

In [ ]:
def build_task_prompt(mode: str, question: str = "") -> str:
    """集中管理四種圖片任務的 prompt，要求模型區分「可見證據」與「推論」。

    參數：mode 為 TASK_MODES 之一；question 只在「圖片問答」使用。
    回傳：送給模型的任務文字。

    TODO（核心技能 2）：為四種模式各寫一段 prompt，至少包含：
      - 圖片描述：先列「直接可見」的物件／文字／場景，再用『可能的推論』標示不確定解讀。
      - 圖片問答：question 去空白後為空 → raise ValueError；只用圖片可見內容當證據，
        無法確認就說不足，不要補造日期、數字、身分。
      - 截圖檢查：依序 (1) 可見文字 (2) 目前狀態 (3) 可能問題 (4) 使用者可自行確認的下一步。
      - 收據／表單抽取：只抽「確實可見」的資料；模糊欄位用 null 並寫進 warnings。
      - 未知 mode → raise ValueError。
    提示：先回傳通用 prompt 讓下游 demo 不會壞；完成後比較四種 prompt 的輸出差異。
    """
    # TODO: 依 mode 回傳對應 prompt
    return "請用繁體中文說明這張圖片中『直接可見』的內容；看不清楚或無法確認時請明確說明。"


print("build_task_prompt 已定義（骨架）")

## 6️⃣ `detail` 是需求選擇，不是品質開關

| detail | 適合 | 注意 |
|---|---|---|
| `low` | 粗略分類、判斷有沒有某物 | 便宜，但小字幾乎讀不到 |
| `high` | 文件、小字、表單、截圖 | 較貴；仍**不保證** OCR、計數或位置正確 |
| `auto` | 不確定時 | 交由模型決定預處理方式 |

<div style="background-color:#fff8e6; border-left:6px solid #f0ad4e; padding:12px 16px; border-radius:6px;">
本教材不宣稱 detail 與成本固定成正比；不同模型支援的 detail 值也可能不同，請依模型文件與實際 <code>usage</code> 檢查。練習 A 會寫一個 <code>choose_detail</code> 規則，讓 App 的成本／品質選擇可以被測試與說明。
</div>

## 7️⃣ 圖片 + Structured Outputs：`RECEIPT_SCHEMA` 與 `analyze_image`

當 UI 必須讀取日期、金額與品項時，不能只要求「請回 JSON」。要提供 **JSON Schema、`strict=True`、
所有必要欄位與 `additionalProperties=False`**（第 6 週技巧）。無法確認的值用 `null`，不要逼模型猜。

`analyze_image` 把整條管線串起來：驗證 → prompt → data URL → request →（抽取模式附 schema）→ API → 分流 → 文字或 dict。
Claude 版多了 `offline=True`：不呼叫 API，回傳**明確標註**的示範樣本，讓沒有金鑰也能走完流程。

<div style="background-color:#eef6ff; border-left:6px solid #4F8BF9; padding:12px 16px; border-radius:6px;">
<b>先看 request 再送出</b>：下一格用 <code>preview_request</code> 印出將送出的結構（data URL 截短、schema 略），這是看懂資料流最快的方式。
</div>

In [ ]:
RECEIPT_SCHEMA = {'type': 'object',
 'properties': {'document_type': {'type': 'string',
                                  'description': '文件類型，例如 receipt、invoice、form 或 unknown。'},
                'merchant': {'type': ['string', 'null'],
                             'description': '可從圖片辨識的商家或機構名稱；無法確認時為 null。'},
                'date': {'type': ['string', 'null'], 'description': '文件日期，保持圖片中的格式；無法確認時為 null。'},
                'currency': {'type': ['string', 'null'], 'description': '可確認的幣別；無法確認時為 null。'},
                'total': {'type': ['number', 'null'], 'description': '可確認的總額數值；不得推測。'},
                'items': {'type': 'array',
                          'items': {'type': 'object',
                                    'properties': {'name': {'type': 'string'},
                                                   'quantity': {'type': ['number', 'null']},
                                                   'amount': {'type': ['number', 'null']}},
                                    'required': ['name', 'quantity', 'amount'],
                                    'additionalProperties': False}},
                'visible_text': {'type': 'array',
                                 'items': {'type': 'string'},
                                 'description': '圖片中確實可讀的重要文字片段。'},
                'warnings': {'type': 'array',
                             'items': {'type': 'string'},
                             'description': '模糊、遮擋、欄位矛盾或無法確認之處。'}},
 'required': ['document_type',
              'merchant',
              'date',
              'currency',
              'total',
              'items',
              'visible_text',
              'warnings'],
 'additionalProperties': False}


def build_request(
    file_bytes: bytes,
    *,
    mode: str,
    question: str = "",
    detail: str = "auto",
    model: str | None = None,
) -> dict[str, Any]:
    """驗證圖片並組出 `client.responses.create(**request)` 要用的參數 dict（不呼叫 API）。"""
    if detail not in DETAIL_LEVELS:
        raise ValueError("detail 只接受 low、high 或 auto。")
    metadata = validate_image(file_bytes)
    prompt = build_task_prompt(mode, question)
    data_url = image_to_data_url(file_bytes, metadata["mime_type"])

    request: dict[str, Any] = {
        "model": model or get_secret("OPENAI_MODEL", DEFAULT_MODEL),
        "instructions": SYSTEM_INSTRUCTIONS,
        "input": [
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": data_url, "detail": detail},
                ],
            }
        ],
        # 不保存本次回應供之後 API 取回；但圖片仍會傳到外部服務處理。
        "store": False,
    }
    if mode == "收據／表單抽取":
        request["text"] = {
            "format": {
                "type": "json_schema",
                "name": "visual_document_result",
                "schema": RECEIPT_SCHEMA,
                "strict": True,
            }
        }
    return request


def preview_request(request: dict[str, Any], max_url_chars: int = 60) -> dict[str, Any]:
    """回傳把 Base64 data URL 截短後的 request 複本，方便印出來看結構而不洗版。"""
    preview = json.loads(json.dumps(request, ensure_ascii=False))
    for message in preview.get("input", []):
        for item in message.get("content", []):
            url = item.get("image_url")
            if isinstance(url, str) and len(url) > max_url_chars:
                item["image_url"] = f"{url[:max_url_chars]}...（共 {len(url)} 字元）"
    if "text" in preview:
        preview["text"]["format"]["schema"] = "（RECEIPT_SCHEMA，略）"
    return preview


def offline_demo_result(mode: str) -> str | dict[str, Any]:
    """離線示範用的固定樣本；每個結果都明確標註不是模型輸出。"""
    if mode == "收據／表單抽取":
        return {
            "document_type": "receipt",
            "merchant": "DEMO CAFE",
            "date": "2026-09-16",
            "currency": "TWD",
            "total": 300,
            "items": [
                {"name": "Coffee", "quantity": 2, "amount": 160},
                {"name": "Sandwich", "quantity": 1, "amount": 95},
                {"name": "Notebook", "quantity": 1, "amount": 45},
            ],
            "visible_text": ["DEMO CAFE - CLASSROOM FIXTURE", "TOTAL TWD 300"],
            "warnings": [OFFLINE_DEMO_NOTICE],
        }
    samples = {
        "圖片描述": "可見：白底、黑色英文文字、數行金額。可能的推論：這是一張收據。",
        "圖片問答": "圖片中可見 TOTAL TWD 300；其他資訊無法從圖片確認。",
        "截圖檢查": "(1) 可見文字：略 (2) 狀態：略 (3) 可能問題：略 (4) 下一步：請自行確認。",
    }
    return f"{samples.get(mode, '（無樣本）')}\n\n{OFFLINE_DEMO_NOTICE}"


def analyze_image(
    file_bytes: bytes,
    *,
    mode: str,
    question: str = "",
    detail: str = "auto",
    model: str | None = None,
    offline: bool = False,
) -> str | dict[str, Any]:
    """驗證圖片後呼叫 Responses API，回傳文字或結構化抽取結果。

    參數：
        file_bytes: 圖片 bytes。
        mode: TASK_MODES 之一；抽取模式會附 JSON Schema。
        question: 問答模式的問題。
        detail: "auto" / "low" / "high"。
        model: 覆蓋預設模型。
        offline: True 時不呼叫 API，只跑驗證與 prompt 組裝，回傳標註過的示範樣本。

    回傳：
        文字（描述／問答／截圖）或 dict（抽取模式）。

    可能錯誤：
        ValueError: 圖片或參數不合法。
        RuntimeError: 缺金鑰、拒答、未完成、空輸出、JSON 解析失敗。

    教學重點：
        `store=False` 只表示不保存回應供後續取回，圖片仍會送到外部服務；
        因此課堂只用自製或明確授權、且不含敏感資料的圖片。
    """
    request = build_request(file_bytes, mode=mode, question=question, detail=detail, model=model)
    if offline:
        return offline_demo_result(mode)

    client = create_client()
    response = client.responses.create(**request)
    output_text = require_completed_response(response)
    if mode != "收據／表單抽取":
        return output_text
    try:
        return json.loads(output_text)
    except json.JSONDecodeError as exc:
        raise RuntimeError("AI 回傳的結構化資料無法解析，請檢查模型與 schema。") from exc


print("RECEIPT_SCHEMA / build_request / preview_request / analyze_image 已定義")

In [ ]:
# 不呼叫 API：只看「將送出的 request」長什麼樣（data URL 截短、schema 略）。
request = build_request(clear_bytes, mode="收據／表單抽取", detail="high", model=MODEL)
print(json.dumps(preview_request(request), ensure_ascii=False, indent=2))

In [ ]:
if RUN_PAID_API:
    print("=== 圖片描述（detail=auto）===")
    print(analyze_image(clear_bytes, mode="圖片描述", detail="auto", model=MODEL))
    print("\n=== 收據抽取（detail=high, JSON Schema）===")
    receipt_result = analyze_image(clear_bytes, mode="收據／表單抽取", detail="high", model=MODEL)
    print(json.dumps(receipt_result, ensure_ascii=False, indent=2))
else:
    print("（未呼叫付費 API）以下為離線示範樣本，只驗證流程：\n")
    print(analyze_image(clear_bytes, mode="圖片描述", offline=True))
    print()
    print(json.dumps(analyze_image(clear_bytes, mode="收據／表單抽取", offline=True), ensure_ascii=False, indent=2))

## 8️⃣ 失敗案例：模糊收據應該得到什麼？

Vision 評估至少要包含：一組清楚圖片、一組小字／模糊圖片、一組「圖片裡沒有答案」的問題。
下一格用模糊旋轉版收據示範：好的結果是 **欄位變 null、warnings 說明模糊**；壞的結果是模型仍自信地給出完整數字。

<div style="background-color:#fdeeee; border-left:6px solid #d9534f; padding:12px 16px; border-radius:6px;">
<b>規則能檢查格式，不能證明看懂</b>：練習 C 的 <code>evaluate_vision_answer</code> 只檢查回答有沒有提到證據與不確定性；真正品質仍需<b>人工對照原圖</b>。
</div>

In [ ]:
display(Image.open(BLURRY_PATH).resize((470, 550)))

if RUN_PAID_API:
    blurry_result = analyze_image(blurry_bytes, mode="收據／表單抽取", detail="high", model=MODEL)
    print(json.dumps(blurry_result, ensure_ascii=False, indent=2))
    print("\n觀察：哪些欄位變成 null？warnings 有沒有說明模糊或旋轉？跟清楚版比對，哪些數字讀錯了？")
else:
    print("（未呼叫付費 API）開啟 RUN_PAID_API 後，比較清楚版與模糊版的 total / items / warnings。")
    print("預期：模糊版應出現 null 或 warnings；若模型仍給出完整數字，更要回看原圖核對。")

In [ ]:
def run_local_checks() -> None:
    """不呼叫付費 API，檢查驗證、prompt、data URL、request 預覽與離線示範。"""
    meta = validate_image(clear_bytes)
    assert meta["mime_type"] == "image/png" and meta["width"] == 900 and meta["height"] == 1100, "metadata 應正確"

    for bad in (b"", b"not an image", b"x" * (MAX_FILE_BYTES + 1)):
        try:
            validate_image(bad)
        except ValueError:
            pass
        else:
            raise AssertionError("空檔／假圖／過大檔應 raise ValueError")

    prompts = {m: build_task_prompt(m, question="總額？") for m in TASK_MODES}
    assert len(set(prompts.values())) == len(TASK_MODES), "四種模式的 prompt 應不同"
    try:
        build_task_prompt("圖片問答", question="   ")
    except ValueError:
        pass
    else:
        raise AssertionError("問答模式沒有問題應 raise ValueError")

    assert image_to_data_url(clear_bytes, "image/png").startswith("data:image/png;base64,"), "data URL 前綴"

    req = build_request(clear_bytes, mode="收據／表單抽取", detail="high", model=MODEL)
    assert req["text"]["format"]["strict"] is True, "抽取模式應附 strict schema"
    assert "..." in json.dumps(preview_request(req), ensure_ascii=False), "預覽應截短 data URL"

    offline = analyze_image(clear_bytes, mode="圖片問答", question="總額？", offline=True)
    assert OFFLINE_DEMO_NOTICE in offline, "離線結果應標註示範"

    print("✅ 本機檢查通過：未呼叫付費 API")


# 這格是「自我檢查」：請先完成上面兩個核心 TODO，再取消下一行註解執行。
# 若在完成前就執行，會因斷言失敗而中斷（這是預期行為，不是程式壞掉）。
# run_local_checks()

## 9️⃣ 從 Notebook 到 Streamlit 專案

本週配套專案：`week12/week12_vision_app_claude/`

| Notebook 概念 | 專案位置 |
|---|---|
| `detect_image_mime` / `validate_image` / `image_to_data_url` | `vision_utils.py` |
| `build_task_prompt` / `choose_detail` / `RECEIPT_SCHEMA` | `vision_utils.py` |
| `build_request` / `preview_request` / `analyze_image`（含 offline） | `vision_utils.py` |
| `normalize_receipt` / `evaluate_vision_answer` | `vision_utils.py` |
| 上傳、本機檢查、表單、結果顯示 | `app.py` |
| 虛構測試收據（清楚版 + 模糊版） | `sample_data/create_demo_receipt.py` |

執行：

```bash
cd week12/week12_vision_app_claude
pip install -r requirements.txt
python sample_data/create_demo_receipt.py
streamlit run app.py
```

> 專案 helper 命名與本 Notebook 一致（且與 Codex 版 `week12_vision_app/` 對齊）。App 側邊欄有「離線示範模式」與 detail 選擇；**API 只在按下「分析圖片」後呼叫**，結果存在 session state。

## ✍️ 課堂練習 A：`choose_detail` 規則（必做）

依任務文字回傳 `"low"`、`"high"` 或 `"auto"`。規則的目的不是取代模型文件，而是讓 App 的成本／品質選擇**可以被測試與說明**。

<div style="background-color:#fff8e6; border-left:6px solid #f0ad4e; padding:12px 16px; border-radius:6px;">
<b>驗收條件</b>：「讀取收據小字」→ high；「粗略分類這張照片」→ low；「一般圖片問答」→ auto。
</div>

In [ ]:
def choose_detail(task: str) -> str:
    """練習 A：依任務文字建議 detail（"low" / "high" / "auto"）。

    TODO：
      1. task 去空白、轉小寫。
      2. 含「收據、表單、小字、截圖、文件、發票、ocr」任一 → "high"。
      3. 含「粗略、大意、分類、有沒有」任一 → "low"。
      4. 其餘 → "auto"。
    這是課堂規則，不是模型品質保證；目的是讓成本／品質選擇可以被測試與說明。
    """
    # TODO: 實作規則
    raise NotImplementedError("請完成 choose_detail")


# 完成後取消註解驗收。
# assert choose_detail("讀取收據小字") == "high"
# assert choose_detail("粗略分類這張照片") == "low"
# assert choose_detail("一般圖片問答") == "auto"
# print("choose_detail 通過")

## ✍️ 課堂練習 B：`normalize_receipt` 正規化（必做）

外部回應進入 UI 前仍要整理資料結構：缺清單改成空 list、缺單值保留 `None`，**不可偷偷用假資料補齊**。
App 的 `render_receipt` 會先經過這個函式再顯示。

In [ ]:
def normalize_receipt(result: dict) -> dict:
    """練習 B：把抽取結果整理成 UI 可安全讀取的固定結構。

    TODO：
      1. document_type 缺少時用 "unknown"。
      2. merchant / date / currency / total 用 result.get() 取出，缺少就保留 None（不可補假值）。
      3. items / visible_text / warnings 缺少或為 None 時改成空 list。
      4. 回傳包含以上 8 個 key 的 dict。
    """
    # TODO: 實作正規化
    raise NotImplementedError("請完成 normalize_receipt")


# 完成後取消註解驗收。
# fake = {"merchant": "DEMO CAFE", "date": "2026-09-16", "currency": "TWD", "total": 300, "items": None}
# normalized = normalize_receipt(fake)
# assert normalized["items"] == [] and normalized["warnings"] == [] and normalized["document_type"] == "unknown"
# print(json.dumps(normalized, ensure_ascii=False, indent=2))

## ✍️ 課堂練習 C：`evaluate_vision_answer` 基本檢查（必做）

用簡單關鍵詞檢查答案是否提到圖片證據、是否標示不確定性，並固定提醒人工覆核。
這種規則只能找出部分格式問題，不能證明答案正確。

In [ ]:
def evaluate_vision_answer(answer: str) -> dict:
    """練習 C：用可解釋的關鍵詞規則檢查回答是否標示證據與不確定性。

    TODO：
      1. answer 去空白 → cleaned。
      2. not_empty：cleaned 非空。
      3. mentions_evidence：含「可見、圖片中、畫面中、文字顯示」任一。
      4. marks_uncertainty：含「無法確認、可能、看不清楚、資訊不足」任一。
      5. needs_human_review 固定 True（規則只能找格式問題，不能證明答案正確）。
    """
    # TODO: 實作三項規則檢查
    raise NotImplementedError("請完成 evaluate_vision_answer")


# 完成後取消註解驗收。
# sample_answer = "圖片中可見 TOTAL TWD 300；日期可能是 2026-09-16，但仍需人工核對。"
# evaluation = evaluate_vision_answer(sample_answer)
# assert evaluation["mentions_evidence"] and evaluation["marks_uncertainty"] and evaluation["needs_human_review"]
# print(evaluation)

## 🚀 課堂練習 D：圖片理解 App 改造（挑戰，選做）

打開 `week12_vision_app_claude/app.py`，從下列方向**選一項**：

1. **多圖比較**：同時上傳兩張截圖，讓模型列出可見差異（`content` 裡放兩個 `input_image`）。
2. **批次抽取**：一次上傳多張收據，逐張抽取後合併成表格並可下載 CSV。
3. **detail 對照**：同一張圖用 `low` 與 `high` 各跑一次，並排顯示答案與 `usage`。
4. **自訂 schema**：把 `RECEIPT_SCHEMA` 改成另一種表單（例如報名表、名片），欄位仍要允許 `null`。

先用下一格把計畫寫成結構化 dict，再動手改；完成後在 README 記錄功能與測試。

In [ ]:
# 練習 D：先規劃再動手。把你的改造計畫填進這個 dict。
challenge_plan = {
    "feature": "",       # TODO: 你要新增的功能名稱
    "input": "",         # TODO: 需要什麼輸入（圖片？多張圖？問題？）
    "output": "",        # TODO: 會產生什麼輸出
    "error_cases": [],   # TODO: 至少列兩個要處理的錯誤情況
    "manual_test": "",   # TODO: 你會怎麼手動測試（用哪張無個資圖片）
}
print(json.dumps(challenge_plan, ensure_ascii=False, indent=2))

## ✅ 完成檢核

- [ ] 能說明 `input_text` + `input_image` 的 content 結構，以及為何選 Base64 data URL。
- [ ] `validate_image` 能擋下空檔、假副檔名、過大檔與動態 GIF。
- [ ] 能把圖片 bytes 轉成 data URL，且不把完整 Base64 印出或存進 log。
- [ ] `build_task_prompt` 四種模式的 prompt 都要求區分證據與推論、看不清楚就說。
- [ ] 能用 `preview_request` 看懂將送出的 request 結構。
- [ ] `run_local_checks()` 不需 API key 即可通過。
- [ ] 用虛構收據啟動 App，先用離線模式測流程，再決定是否開啟付費 API。
- [ ] 記錄一個 Vision 看錯的案例，說明如何以 UI、prompt 或人工覆核降低風險。
- [ ] API key 未寫進程式碼、Notebook 或 Git。
- [ ] 完成至少練習 A、B、C。

## ❓ 常見問題

**模型把數字讀錯了怎麼辦？**
改用 `detail="high"`、裁切只留關鍵區域、或先用 Pillow 放大。但任何 detail 都不保證 OCR 正確；金額類一定人工核對。

**可以用圖片 URL 嗎？**
可以（`image_url` 直接放公開網址），但課堂圖片不該公開；Streamlit 上傳給的是 bytes，用 data URL 最單純。

**抽取回來的 JSON 解析失敗？**
先確認有附 `text.format` 的 JSON Schema 且 `strict=True`；再看 `require_completed_response` 是否因未完成（長度限制）而被截斷。

**`store=False` 就代表圖片不會被處理嗎？**
不是。它只表示不保存回應供後續 API 取回；圖片仍會送到外部服務處理，所以才禁止上傳敏感圖片。

**離線示範模式的結果可以拿來評估嗎？**
不行。它是固定樣本，只用來驗證流程與畫面；評估一定要用真 API 並人工對照原圖。

**多張圖片怎麼送？**
同一個 `content` 陣列放多個 `input_image` 即可；每張都計費，張數越多成本越高。

## 📝 課後任務

用**自己製作或明確授權、不含敏感資料**的圖片：

1. 準備三張圖：一張清楚、一張小字或模糊、一張「問題在圖裡找不到答案」的。
2. 每張各跑一次描述與問答，記錄答案、`detail` 與是否正確標示「無法確認」。
3. 用虛構收據跑抽取，對照原圖核對每個欄位；記錄至少一個看錯的案例與你的因應方式。
4. 用 `evaluate_vision_answer` 檢查回答格式，並補一段人工評分（有無證據／有無亂猜／答案正確各 0/1）。
5. 把專案與 README 推送到自己的 GitHub repo（確認 `.env` 與生成的 PNG 未被追蹤）。

## 🔍 補充：Claude API 的圖片輸入（選讀，不需執行）

課程主線是 OpenAI；期末或自主學習週若想比較，Claude API 的圖片輸入結構如下（同樣是「文字 + 圖片」放在同一則訊息）：

| | OpenAI Responses API | Claude Messages API |
|---|---|---|
| 圖片 content type | `{"type": "input_image", "image_url": data_url, "detail": ...}` | `{"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": <純 base64>}}` |
| 文字 content type | `{"type": "input_text", "text": ...}` | `{"type": "text", "text": ...}` |
| Base64 形式 | 完整 data URL（含 `data:<mime>;base64,` 前綴） | 只放 base64 內容，mime 另寫在 `media_type` |
| 結構化輸出 | `text.format` 放 JSON Schema（strict） | 以 tool use 定義 `input_schema` 強制 JSON |
| detail 參數 | 有 | 無（以圖片尺寸控制） |

> 兩家都會把圖片換算成 token 計費，也都可能看錯小字與數字；驗證、prompt 與人工核對的原則完全相同。

## 🔮 下週預告：Function Calling / Tool Calling

本週的 `validate_image`、`build_task_prompt`、`analyze_image` 已經把任務邏輯拆成**輸入明確、輸出明確、有錯誤處理**的函式。
第 13 週會把這種函式定義成工具（JSON Schema 描述參數），交給模型**自己決定何時呼叫、帶什麼參數**，
再把執行結果送回模型組成最終答案。

> 請保留本週的 helper；下週會直接把它們包成 Skill-like 模組給模型使用。